<a href="https://colab.research.google.com/github/michaelseth5/-Monte-Carlo-Simulation-for-College-Football-Team-Rankings/blob/main/Michael_Katongole_BTT_Accenture_1K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## CUAD clause detection: TF-IDF baseline
This notebook trains a multi-label classifier on Aditi's labeled contract chunks. It needs a ZIP containing `cuad_train_split.jsonl`, `cuad_val_split.jsonl`, and `cuad_test_split.jsonl`. A ZIP containing the original CUAD JSON files is not the processed dataset.
Run the cells in order. Use the validation split to assess and improve the baseline. Keep the test split untouched until the final evaluation.

### 1. Upload the processed splits
In Aditi's notebook, run through the final Splits saved. cell. Download the three JSONL outputs and place them together in a ZIP. Upload that ZIP when prompted below.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files
import pandas as pd

required = ["cuad_train_split.jsonl", "cuad_val_split.jsonl", "cuad_test_split.jsonl"]
print("Please upload the ZIP file containing the processed splits (cuad_train_split.jsonl, cuad_val_split.jsonl, cuad_test_split.jsonl) when prompted.")
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Select one ZIP containing Aditi's three processed split files.")

zip_name = next(iter(uploaded))
data_dir = Path("/content/cuad_processed")
data_dir.mkdir(parents=True, exist_ok=True)

with ZipFile(zip_name) as archive:
    members = {Path(name).name: name for name in archive.namelist()}
    missing = [name for name in required if name not in members]
    if missing:
        raise ValueError(
            f"Missing processed files: {missing}. ZIP contains: {list(members)}. "
            "Run Aditi's notebook through 'Splits saved.' and ZIP its output JSONL files."
        )
    for name in required:
        (data_dir / name).write_bytes(archive.read(members[name]))

train_chunks = pd.read_json(data_dir / required[0], lines=True)
val_chunks = pd.read_json(data_dir / required[1], lines=True)
test_chunks = pd.read_json(data_dir / required[2], lines=True)

for name, frame in [("train", train_chunks), ("validation", val_chunks), ("test", test_chunks)]:
    needed = {"contract_id", "chunk_text", "labels"}
    assert needed.issubset(frame.columns), f"{name} missing columns: {needed - set(frame.columns)}"
    assert frame["labels"].map(lambda x: isinstance(x, list)).all(), f"{name} labels must be lists"
    print(name, "chunks:", len(frame), "contracts:", frame["contract_id"].nunique())

display(train_chunks[["contract_id", "chunk_text", "labels"]].head(3))

Please upload the ZIP file containing the processed splits (cuad_train_split.jsonl, cuad_val_split.jsonl, cuad_test_split.jsonl) when prompted.


### 2. Check that the data is suitable for evaluation
One contract must belong to only one split. A chunk may have zero, one, or several clause labels.

In [ ]:
train_ids = set(train_chunks["contract_id"])
val_ids = set(val_chunks["contract_id"])
test_ids = set(test_chunks["contract_id"])

assert not (train_ids & val_ids), "Train and validation share a contract"
assert not (train_ids & test_ids), "Train and test share a contract"
assert not (val_ids & test_ids), "Validation and test share a contract"

if "token_count" in train_chunks:
    assert max(df["token_count"].max() for df in [train_chunks, val_chunks, test_chunks]) <= 512

print("No contracts shared between splits.")
print("Training chunks by number of labels:")
print(train_chunks["labels"].map(len).value_counts().sort_index().head(12))
assert train_chunks["labels"].map(len).eq(0).any(), "No unlabeled training chunks: inspect processing"
assert train_chunks["labels"].map(len).gt(0).any(), "No positive training chunks: inspect processing"

### 3. Choose clause categories with training examples
CUAD uses names such as Cap On Liability and Uncapped Liability. The notebook prints unavailable or rare categories rather than inventing labels for them. These are clause-presence labels, not Low/Medium/High risk ratings.

In [ ]:
from collections import Counter

priority_categories = [
    "Limitation Of Liability", "Cap On Liability", "Uncapped Liability",
    "Indemnification", "Termination For Convenience", "Governing Law",
    "Renewal Term", "Ip Ownership Assignment", "Change Of Control",
    "Non-Compete", "Audit Rights", "Most Favored Nation",
]

train_counts = Counter(label for labels in train_chunks["labels"] for label in labels)
val_counts = Counter(label for labels in val_chunks["labels"] for label in labels)
coverage = pd.DataFrame({
    "category": priority_categories,
    "train_positive_chunks": [train_counts[c] for c in priority_categories],
    "val_positive_chunks": [val_counts[c] for c in priority_categories],
})
display(coverage)

active_categories = [c for c in priority_categories if train_counts[c] >= 5]
if not active_categories:
    raise ValueError("No priority categories with at least 5 positive training chunks. Check label names.")
print("Categories to model:", active_categories)
print("Skipped categories:", [c for c in priority_categories if c not in active_categories])

### 4. Train TF-IDF + logistic regression
The vectorizer learns vocabulary from training chunks only. The classifier learns a separate yes/no decision for each clause category.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MultiLabelBinarizer

label_encoder = MultiLabelBinarizer(classes=active_categories)
label_encoder.fit([[]])
selected = set(active_categories)
y_train = label_encoder.transform([[c for c in labels if c in selected] for labels in train_chunks["labels"]])
y_val = label_encoder.transform([[c for c in labels if c in selected] for labels in val_chunks["labels"]])

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000, sublinear_tf=True),
    OneVsRestClassifier(LogisticRegression(class_weight="balanced", max_iter=1000))
)
model.fit(train_chunks["chunk_text"].fillna(""), y_train)
print("Model trained on", len(train_chunks), "chunks.")

### 5. Evaluate on contracts the model has not seen
support is the number of positive validation chunks. Watch recall for important clauses, and interpret scores for categories with few examples cautiously.

In [ ]:
from sklearn.metrics import classification_report

predictions = model.predict(val_chunks["chunk_text"].fillna(""))
report = classification_report(
    y_val, predictions, target_names=label_encoder.classes_, zero_division=0,
    output_dict=True
)
results = pd.DataFrame(report).T
display(results.round(3))

per_category = results.loc[list(label_encoder.classes_), ["precision", "recall", "f1-score", "support"]]
per_category.to_csv("cuad_tfidf_validation_metrics.csv")
print("Saved per-category validation results: cuad_tfidf_validation_metrics.csv")

### 6. Review a few errors
Read the clause text for a few false positives and missed clauses. This makes the baseline useful for deciding what to improve.

In [ ]:
import numpy as np

category = active_categories[0]  # Change this to any printed active category
idx = list(label_encoder.classes_).index(category)
false_negatives = np.flatnonzero((y_val[:, idx] == 1) & (predictions[:, idx] == 0))
false_positives = np.flatnonzero((y_val[:, idx] == 0) & (predictions[:, idx] == 1))

print("Category:", category)
print("Missed positive chunks:", len(false_negatives))
print("Incorrectly flagged chunks:", len(false_positives))
for name, positions in [("Missed clause", false_negatives), ("Incorrect flag", false_positives)]:
    for pos in positions[:2]:
        row = val_chunks.iloc[pos]
        print("\n ", name, "| contract:", row["contract_id"])
        print("True labels:", row["labels"])
        print("Text:", row["chunk_text"][:500])

### What to do after this runs
1. Confirm the split counts, leakage check, and per-category metrics appear without errors.
2. Report validation precision, recall, F1, and support for each modeled clause category. Explain any categories that were unavailable or rare.
3. Review false positives and false negatives, then make one justified improvement if needed.
4. Evaluate on `test_chunks` only after model and thresholds are fixed.
5. Save this notebook and its metrics file to the team repository.

In [ ]:
import json
import pandas as pd
import json
import re
from collections import defaultdict

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
!wget -q https://raw.githubusercontent.com/johnhallman/cuad-data/main/data/train_separate_questions.json -O train_separate_questions.json
!wget -q https://raw.githubusercontent.com/johnhallman/cuad-data/main/data/test.json -O test.json
!wget -q https://raw.githubusercontent.com/johnhallman/cuad-data/main/data/category_descriptions.csv -O category_descriptions.csv

print("Data files downloaded.")

Data files downloaded.


In [ ]:
# from google.colab import files
# from zipfile import ZipFile
# from pathlib import Path
# import pandas as pd

# uploaded = files.upload()  # Choose cuad_model_data.zip from your computer
# zip_name = next(iter(uploaded))

# data_dir = Path("/content/cuad_model_data")
# data_dir.mkdir(exist_ok=True)

# with ZipFile(zip_name) as archive:
#     archive.extractall(data_dir)

# train_chunks = pd.read_json(data_dir / "cuad_train_split.jsonl", lines=True)
# val_chunks = pd.read_json(data_dir / "cuad_val_split.jsonl", lines=True)
# test_chunks = pd.read_json(data_dir / "cuad_test_split.jsonl", lines=True)

# print("Train:", train_chunks.shape)
# print("Validation:", val_chunks.shape)
# print("Test:", test_chunks.shape)
# display(train_chunks[["contract_id", "chunk_text", "labels"]].head())

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from google.colab import files

names = [
    "train_separate_questions.json",
    "test.json",
    "category_descriptions.csv",
]

for name in names:
    assert Path(name).exists(), f"Missing: {name}"

with ZipFile("cuad_model_data.zip", "w") as archive:
    for name in names:
        archive.write(name)

files.download("cuad_model_data.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# !git clone -b main https://github.com/Break-Through-Tech/Accenture-1K-contract-review-challenge.git /content/team_repo

# df = !git clone -b main https://github.com/Break-Through-Tech/Accenture-1K-contract-review-challenge.git /content/team_repo
# print(df)

In [ ]:
from pathlib import Path
import pandas as pd

repo = Path("Accenture-1K-contract-review-challenge")

# See which CSV files are in the repository
csv_files = list(repo.rglob("*.csv"))
print(csv_files)

# Replace this with the CSV you want from the printed list
df = pd.read_csv(csv_files[0])
print(df.head())
print(df.shape)

[PosixPath('Accenture-1K-contract-review-challenge/data/category_descriptions.csv')]
  Category (incl. context and answer)  \
0             Category: Document Name   
1                   Category: Parties   
2            Category: Agreement Date   
3            Category: Effective Date   
4           Category: Expiration Date   

                                         Description  \
0              Description: The name of the contract   
1  Description: The two or more parties who signe...   
2              Description: The date of the contract   
3  Description: The date when the contract is eff...   
4  Description: On what date will the contract's ...   

                                  Answer Format     Group  
0                  Answer Format: Contract Name  Group: -  
1     Answer Format: Entity or individual names  Group: -  
2              Answer Format: Date (mm/dd/yyyy)  Group: 1  
3              Answer Format: Date (mm/dd/yyyy)  Group: 1  
4  Answer Format: Date (mm/dd/y

In [ ]:
display(df.head())
print(df.columns.tolist())
print(df.shape)


,Category (incl. context and answer),Description,Answer Format,Group
0,Category: Document Name,Description: The name of the contract,Answer Format: Contract Name,Group: -
1,Category: Parties,Description: The two or more parties who signe...,Answer Format: Entity or individual names,Group: -
2,Category: Agreement Date,Description: The date of the contract,Answer Format: Date (mm/dd/yyyy),Group: 1
3,Category: Effective Date,Description: The date when the contract is eff...,Answer Format: Date (mm/dd/yyyy),Group: 1
4,Category: Expiration Date,Description: On what date will the contract's ...,Answer Format: Date (mm/dd/yyyy) / Perpetual,Group: 1


['Category (incl. context and answer)', 'Description', 'Answer Format', 'Group']
(41, 4)


In [ ]:
from pathlib import Path

repo = Path("Accenture-1K-contract-review-challenge")
for path in repo.rglob("*"):
    if path.is_file() and ".git" not in path.parts:
        print(path)

Accenture-1K-contract-review-challenge/.gitignore
Accenture-1K-contract-review-challenge/Challenge-Project-Overview.md
Accenture-1K-contract-review-challenge/requirements.txt
Accenture-1K-contract-review-challenge/Getting-Started-for-Fellows.md
Accenture-1K-contract-review-challenge/README.md
Accenture-1K-contract-review-challenge/data/.gitkeep
Accenture-1K-contract-review-challenge/data/category_descriptions.csv
Accenture-1K-contract-review-challenge/data/cuad-v1.zip
Accenture-1K-contract-review-challenge/data/README.md
Accenture-1K-contract-review-challenge/data/test.json
Accenture-1K-contract-review-challenge/data/CUADv1.json
Accenture-1K-contract-review-challenge/data/train_separate_questions.json
Accenture-1K-contract-review-challenge/notebooks/.gitkeep


In [ ]:
display(df.head(200))




,Category (incl. context and answer),Description,Answer Format,Group
0,Category: Document Name,Description: The name of the contract,Answer Format: Contract Name,Group: -
1,Category: Parties,Description: The two or more parties who signe...,Answer Format: Entity or individual names,Group: -
2,Category: Agreement Date,Description: The date of the contract,Answer Format: Date (mm/dd/yyyy),Group: 1
3,Category: Effective Date,Description: The date when the contract is eff...,Answer Format: Date (mm/dd/yyyy),Group: 1
4,Category: Expiration Date,Description: On what date will the contract's ...,Answer Format: Date (mm/dd/yyyy) / Perpetual,Group: 1
5,Category: Renewal Term,Description: What is the renewal term after th...,Answer Format: [Successive] number of years/mo...,Group: 1
6,Category: Notice Period to Terminate Renewal,Description: What is the notice period require...,Answer Format: Number of days/months/year(s),Group: 1
7,Category: Governing Law,Description: Which state/country's law governs...,Answer Format: Name of a US State / non-US Pro...,Group: -
8,Category: Most Favored Nation,Description: Is there a clause that if a third...,Answer Format: Yes/No,Group: -
9,Category: Non-Compete,Description: Is there a restriction on the abi...,Answer Format: Yes/No,Group: 2


In [ ]:
import json
from pathlib import Path
import pandas as pd
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# --- Data Loading and Preparation (further corrected) ---
repo_dir = 'Accenture-1K-contract-review-challenge'
train_data_path = Path(repo_dir) / 'data' / 'train_separate_questions.json'
test_data_path = Path(repo_dir) / 'data' / 'test.json' # Assuming test.json is for validation

def load_and_process_data(file_path):
    texts = []
    labels = []
    contract_ids = [] # Added for contract_id
    print(f"Attempting to load data from: {file_path}")
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        print(f"Successfully loaded JSON from {file_path}.")
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return pd.DataFrame({"chunk_text": [], "labels": [], "contract_id": []})
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON from {file_path}. Is it valid JSON?")
        return pd.DataFrame({"chunk_text": [], "labels": [], "contract_id": []})

    if isinstance(data, list):
        print("Detected top-level list structure.")
        for item_idx, item in enumerate(data):
            context = None
            current_labels_set = set()
            contract_id = item.get('title', f"doc_{item_idx}") # Attempt to get contract_id

            context = item.get('context') or item.get('text')

            direct_labels = item.get('labels')
            if isinstance(direct_labels, list) and direct_labels:
                current_labels_set.update(direct_labels)

            if 'qas' in item and isinstance(item['qas'], list):
                for qa_idx, qa in enumerate(item['qas']):
                    qa_labels = qa.get('labels')
                    if isinstance(qa_labels, list) and qa_labels:
                        current_labels_set.update(qa_labels)

                    for ans_idx, ans in enumerate(qa.get('answers', [])):
                        ans_labels = ans.get('labels')
                        if isinstance(ans_labels, list) and ans_labels:
                            current_labels_set.update(ans_labels)
                        entity_type_or_label = ans.get('entity_type') or ans.get('entity_label')
                        if isinstance(entity_type_or_label, str) and entity_type_or_label:
                            current_labels_set.add(entity_type_or_label)

                    for anno_idx, anno in enumerate(qa.get('annotations', [])):
                        anno_labels = anno.get('labels')
                        if isinstance(anno_labels, list) and anno_labels:
                            current_labels_set.update(anno_labels)
                        entity_type_or_label = anno.get('entity_type') or anno.get('entity_label')
                        if isinstance(entity_type_or_label, str) and entity_type_or_label:
                            current_labels_set.add(entity_type_or_label)

            if not current_labels_set and 'question' in item and isinstance(item['question'], str) and item['question'].strip():
                question_label = item['question'].strip()
                if question_label.lower().startswith('what is the '):
                    question_label = question_label[len('what is the '):].strip()
                if question_label.endswith('?'):
                    question_label = question_label[:-1].strip()
                if question_label:
                    current_labels_set.add(question_label)

            if context and current_labels_set:
                texts.append(context)
                labels.append(list(current_labels_set))
                contract_ids.append(contract_id) # Append contract_id

    elif isinstance(data, dict) and 'data' in data and isinstance(data['data'], list):
        print("Detected SQuAD-like dictionary structure with 'data' key.")
        for doc_idx, doc in enumerate(data['data']):
            contract_id = doc.get('title', f"doc_{doc_idx}") # Extract contract_id from title
            for par_idx, paragraph in enumerate(doc.get('paragraphs', [])):
                context = paragraph.get('context')
                qas = paragraph.get('qas', [])
                paragraph_labels_set = set()
                for qa_idx, qa in enumerate(qas):
                    qa_labels = qa.get('labels')
                    if isinstance(qa_labels, list) and qa_labels:
                        paragraph_labels_set.update(qa_labels)

                    for ans_idx, ans in enumerate(qa.get('answers', [])):
                        ans_labels = ans.get('labels')
                        if isinstance(ans_labels, list) and ans_labels:
                            paragraph_labels_set.update(ans_labels)
                        entity_type_or_label = ans.get('entity_type') or ans.get('entity_label')
                        if isinstance(entity_type_or_label, str) and entity_type_or_label:
                            paragraph_labels_set.add(entity_type_or_label)

                    for anno_idx, anno in enumerate(qa.get('annotations', [])):
                        anno_labels = anno.get('labels')
                        if isinstance(anno_labels, list) and anno_labels:
                            paragraph_labels_set.update(anno_labels)
                        entity_type_or_label = anno.get('entity_type') or anno.get('entity_label')
                        if isinstance(entity_type_or_label, str) and entity_type_or_label:
                            paragraph_labels_set.add(entity_type_or_label)

                if not paragraph_labels_set and context and qas:
                    for qa in qas:
                        if 'question' in qa and isinstance(qa['question'], str) and qa['question'].strip():
                            question_label = qa['question'].strip()
                            if question_label.lower().startswith('what is the '):
                                question_label = question_label[len('what is the '):].strip()
                            if question_label.endswith('?'):
                                question_label = question_label[:-1].strip()
                            if question_label:
                                paragraph_labels_set.add(question_label)

                if context and paragraph_labels_set:
                    texts.append(context)
                    labels.append(list(paragraph_labels_set))
                    contract_ids.append(contract_id) # Append contract_id
    else:
        print(f"Warning: Unknown JSON structure for {file_path}. Expected a list or a dict with 'data' key.")

    print(f"Loaded {len(texts)} entries with labels from {file_path}")
    return pd.DataFrame({"chunk_text": texts, "labels": labels, "contract_id": contract_ids})

train_chunks = load_and_process_data(train_data_path)
val_chunks = load_and_process_data(test_data_path)

print(f"Train chunks shape: {train_chunks.shape}")
print(f"Validation chunks shape: {val_chunks.shape}")

label_encoder = MultiLabelBinarizer()

if not train_chunks.empty:
    y_train = label_encoder.fit_transform(train_chunks["labels"])
elif not train_chunks.empty and train_chunks["labels"].apply(lambda x: len(x) > 0).any():
    # Fallback for cases where 'labels' column exists but is empty
    y_train = label_encoder.fit_transform(train_chunks["labels"])
else:
    print("Warning: train_chunks is empty or 'labels' column is empty. Cannot fit MultiLabelBinarizer. Setting y_train to empty list.")
    y_train = []

if not val_chunks.empty and hasattr(label_encoder, 'classes_') and len(label_encoder.classes_) > 0:
    y_val = label_encoder.transform(val_chunks["labels"])
elif not val_chunks.empty and val_chunks["labels"].apply(lambda x: len(x) > 0).any() and hasattr(label_encoder, 'classes_') and len(label_encoder.classes_) > 0:
    # Fallback for cases where 'labels' column exists but is empty
    y_val = label_encoder.transform(val_chunks["labels"])
else:
    print("Warning: val_chunks is empty, 'labels' column is empty, or label_encoder not fitted/has no classes. Cannot transform validation labels. Setting y_val to empty list.")
    y_val = []

if not train_chunks.empty and len(y_train) > 0:
    features = FeatureUnion([
        ("words", TfidfVectorizer(
            ngram_range=(1, 2), min_df=2, max_features=50000
        )),
        ("characters", TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5),
            min_df=2, max_features=50000
        ))
    ])

    model = make_pipeline(
        features,
        OneVsRestClassifier(
            LogisticRegression(class_weight="balanced", max_iter=1000)
        )
    )

    model.fit(train_chunks["chunk_text"].fillna(""), y_train)

    if not val_chunks.empty and len(y_val) > 0:
        predictions = model.predict(val_chunks["chunk_text"].fillna(""))

        print(classification_report(
            y_val,
            predictions,
            target_names=label_encoder.classes_,
            zero_division=0
        ))
    else:
        print("Skipping prediction and classification report due to empty validation data or no classes.")
else:
    print("Skipping model training and evaluation due to empty training data or no labels.")

Attempting to load data from: Accenture-1K-contract-review-challenge/data/train_separate_questions.json
Successfully loaded JSON from Accenture-1K-contract-review-challenge/data/train_separate_questions.json.
Detected SQuAD-like dictionary structure with 'data' key.
Loaded 408 entries with labels from Accenture-1K-contract-review-challenge/data/train_separate_questions.json
Attempting to load data from: Accenture-1K-contract-review-challenge/data/test.json
Successfully loaded JSON from Accenture-1K-contract-review-challenge/data/test.json.
Detected SQuAD-like dictionary structure with 'data' key.
Loaded 102 entries with labels from Accenture-1K-contract-review-challenge/data/test.json
Train chunks shape: (408, 3)
Validation chunks shape: (102, 3)


/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 0 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 1 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 2 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 3 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 4 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 5 is present in all training examples.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/multiclass.py:90: UserWarning: Label 6 is present in all training examples.
  warnings.warn(
/usr/local/li

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        precision    recall  f1-score   support

                                                                                                                                                                                                                                                                       Highlight the parts (if any) of this contract related to "Affiliate License-Licensee" that should be reviewed by a lawyer. Details: Does the contract contain a license grant to

In [ ]:
for path in [train_data_path, test_data_path]:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    print("\nFILE:", path)
    print("Top-level keys:", raw.keys())
    doc = raw["data"][0]
    paragraph = doc["paragraphs"][0]
    qa = paragraph["qas"][0]

    print("Document keys:", doc.keys())
    print("Paragraph keys:", paragraph.keys())
    print("Question keys:", qa.keys())
    print("Example question:", qa["question"])
    print("Example answers:", qa["answers"][:1])


FILE: Accenture-1K-contract-review-challenge/data/train_separate_questions.json
Top-level keys: dict_keys(['version', 'data'])
Document keys: dict_keys(['title', 'paragraphs'])
Paragraph keys: dict_keys(['qas', 'context'])
Question keys: dict_keys(['answers', 'id', 'question', 'is_impossible'])
Example question: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Example answers: [{'text': 'DISTRIBUTOR AGREEMENT', 'answer_start': 44}]

FILE: Accenture-1K-contract-review-challenge/data/test.json
Top-level keys: dict_keys(['version', 'data'])
Document keys: dict_keys(['title', 'paragraphs'])
Paragraph keys: dict_keys(['qas', 'context'])
Question keys: dict_keys(['answers', 'id', 'question', 'is_impossible'])
Example question: Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
Example answers: [{'text': 'S

In [ ]:
print(train_chunks["labels"].apply(len).value_counts())
print(train_chunks[["labels"]].head(1).to_string())

labels
41    408
Name: count, dtype: int64
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [ ]:
print("Training rows:", len(train_chunks))
print("Validation rows:", len(val_chunks))
print("Columns:", train_chunks.columns.tolist())

display(train_chunks[["chunk_text", "labels"]].head())

Training rows: 408
Validation rows: 102
Columns: ['chunk_text', 'labels']


,chunk_text,labels
0,EXHIBIT 10.6\n\n ...,[Highlight the parts (if any) of this contract...
1,Exhibit 10.26 CONFIDENTIAL TREATMENT HAS BE...,[Highlight the parts (if any) of this contract...
2,Exhibit 1\n\nJOINT FILING AGREEMENT\n\nThe und...,[Highlight the parts (if any) of this contract...
3,REDACTED COPY\n\nCONFIDENTIAL TREATMENT REQUES...,[Highlight the parts (if any) of this contract...
4,Exhibit 10.23 Corporate Address Fannin South P...,[Highlight the parts (if any) of this contract...


In [ ]:
overlap = set(train_chunks["contract_id"]) & set(val_chunks["contract_id"])
print("Contracts in both sets:", len(overlap))  # Must be 0

Contracts in both sets: 0


In [ ]:
from pathlib import Path

repo = Path("/content/team_repo")
for name in ["cuad_train_split.jsonl", "cuad_val_split.jsonl", "cuad_test_split.jsonl"]:
    print(name, list(repo.rglob(name)))